# Real-Time Anomaly Detection in Big Data Streams Using LSTM Autoencoders
### A Distributed Processing Approach

**Student ID:** 2026257
**Module:** Advanced Data Analytics / Big Data Storage and Processing, Integrated CA1 Sem 2
**Dataset:** Credit Card Fraud Detection (Kaggle, mlg-ulb/creditcardfraud)



## 0. Environment Setup

In [1]:
# !pip install pyspark==3.5.1 tensorflow scikit-learn pandas matplotlib seaborn kagglehub --quiet
print("Skip re-running if packages are already installed.")


Skip re-running if packages are already installed.


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, roc_auc_score, roc_curve, f1_score
)

print("PySpark:", pyspark.__version__, "| TensorFlow:", tf.__version__)


PySpark: 4.0.4 | TensorFlow: 2.21.0


## 1. Initialize Spark Session (Big Data Component)

In [3]:
spark = (
    SparkSession.builder
    .appName("CA1_BigData_NeuralNetworks")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark


## 2. Load Dataset


In [4]:
import os, glob

candidates = glob.glob("*.csv") + glob.glob("/content/*.csv") + glob.glob("/content/sample_data/*.csv")
candidates = [c for c in candidates if "sample_data" not in c or "creditcard" in c.lower()]

if candidates:
    csv_path = candidates[0]
    print("Auto-detected dataset file:", csv_path)
else:
    csv_path = "/content/creditcard.csv"
    print("No CSV auto-detected -- make sure 'creditcard.csv' is uploaded, or set csv_path manually.")

df_spark = spark.read.csv(csv_path, header=True, inferSchema=True)
df_spark.printSchema()
print("Row count:", df_spark.count())
df_spark.show(5)


Auto-detected dataset file: creditcard.csv
root
 |-- Time: integer (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: doub